# Notebook 06 - Churn Prediction (Classification)

**Input:** `data/processed/customer_segments.parquet` (5,265 customers x 45 columns)<br>
**Output:**
- `data/processed/churn_predictions.parquet` - each customer with predicted churn probability
- `models/churn_xgboost.joblib` - trained model + metadata
- `reports/figures/churn_*.png` - pitch-ready visualizations

## How this notebook differs from Notebook 05
Notebook 05 predicted **how much** revenue (continous -> regression). Notebook 06 predicts **whether** a customer will buy at all (binary -> classification).<br>
Same modeling framework, different problem shape and different evaluation metrics.

**Why we want both models:**
- CLV regression tells you a customer is worth £450, but doesn't directly answer "will they buy?"
- Churn classification tells you they have 78% probability of buying, but not how much
- Combined: you get the 2x2 matrix that drives the recommendation engine

## Target setup:
- `target_purchased_90d = 1` -> customer bought in next 90 days (active)
- `target_purchased_90d = 0` -> customer did NOT buy (churned)

So we're literally predicting `P(active)`. Churn probability = `1 - P(active)`.

**Class balance:** 43.5% active / 56.5% churned. Mildly imbalanced - we'll use `scale_pos_weight` defensively but it isn't severe enough to require SMOTE or undersampling.

## Notebook structure:
1. Load and prepare features + target
2. Train/test split (stratified on the target)
3. Baseline classifier (majority class)
4. XGBoost classifier
5. Evaluation - ROC, AUC, PR AUC, confusion matrix
6. Probability calibration check
7. Threshold selection - business-driven, not arbitrary
8. SHAP explanations
9. Predictions on full dataset, build the 2x2 strategy matrix with CLV
10. Save model + predictions

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report, f1_score
)
from sklearn.calibration import calibration_curve
from sklearn.dummy import DummyClassifier
import xgboost as xgb
import shap

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
sns.set_style('whitegrid')

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
REPORTS_DIR = Path('../reports/figures')
MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE=42

## 1. Load data and prepare features
Same feature setup as Notebook 05 - we exclude IDs, raw country, and the targets themselves. The features that worked for CLV regression will also work for churn classification because they encode the same underlying customer behavior.

In [5]:
df = pd.read_parquet(PROCESSED_DIR / 'customer_segments.parquet')
print(f'Loaded {len(df):,} customers x {df.shape[1]} columns')

EXCLUDE_COLS = {
    'CustomerID', 'primary_country', 'RFM_score', 'cluster_name',
    'target_revenue_90d', 'target_orders_90d', 'target_purchased_90d'
}

df_encode = pd.get_dummies(df, columns=['segment'], prefix='segment')

feature_cols = [c for c in df_encode.columns if c not in EXCLUDE_COLS]
X = df_encode[feature_cols].copy()
y = df_encode['target_purchased_90d'].copy()

print(f'Features: {X.shape[1]} columns')
print(f'Class balance: {y.mean()*100:.1f}% positive (active), {(1-y.mean())*100:.1f}% negative (churned)')

Loaded 5,256 customers x 45 columns
Features: 46 columns
Class balance: 43.6% positive (active), 56.4% negative (churned)


## 2. Train/test split - stratified on target
Stratification is essential for classification: it guarantees the train and test sets have the same class balance.<br>
Without it, you could randomly land 60% positives in train and 30% in test, throwing off the evaluation.

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE
)

print(f'Train: {len(X_train):,} ({y_train.mean()*100:.1f}% positive)')
print(f'Test: {len(X_test):,} ({y_test.mean()*100:.1f}% positive)')

Train: 4,204 (43.6% positive)
Test: 1,052 (43.5% positive)


## 3. Baseline - what does "dumb" look like?
For classification, two common baselines:
- **Majority class:** always predict the most common class (here:0/churned). Accuracy = 56.5%
- **Stratified:** predict randomly proportional to class frequencies. Random performance.

We use majority class - it sets the floor. Any usable model must beat this meaningfully.

In [10]:
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)
y_proba_baseline = baseline.predict_proba(X_test)[:,1]

print(f'Baseline (predict majority class for everyone):')
print(f'    Accuracy: {(y_pred_baseline == y_test).mean():.3f}')
print(f'    ROC AUC: {roc_auc_score(y_test, y_proba_baseline):.3f} (chance = 0.5)')
print(f'    F1: {f1_score(y_test, y_pred_baseline, zero_division=0):.3f}')
print('\nThis is the floor. Our model must beat 0.5 ROC AUC meaningfully.')

Baseline (predict majority class for everyone):
    Accuracy: 0.565
    ROC AUC: 0.500 (chance = 0.5)
    F1: 0.000

This is the floor. Our model must beat 0.5 ROC AUC meaningfully.
